In [ ]:
# ensure connection to cluster
spark

In [1]:
# ─── SPARK GTEx PIPELINE ─────────────────────────────────────────────────────
# Replaces the pandas pd.read_parquet approach that OOMs on 32 GB RAM.
# Uses Spark lazy evaluation + row-wise mapInPandas to avoid loading the
# full 74K x 19K matrix (~11 GB) into driver memory at once.
#
# Produces the same variable names as the existing cells so ANOVA / RF /
# heatmap cells work unchanged.  After running this cell, skip to Cell E.
#
#   log2fc_by_treatment, log2fc_filtered
#   sample_to_tissue, sample_to_donor, donor_groups
#   impact_by_donor, stacked, tissue_cols
#   impact_agg  (for the aggregated-analysis section)
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from collections import defaultdict

# ── 1. Tissue / donor metadata ──────────────────────────────────────
attrs_pd = pd.read_csv(
    "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/"
    "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
    sep="\t", usecols=["SAMPID", "SMTSD"]
)
attrs_pd["SUBJID"]       = attrs_pd["SAMPID"].str.split("-").str[:2].str.join("-")
attrs_pd["tissue_group"] = attrs_pd["SMTSD"].str.split(" - ").str[0].str.strip()
sample_to_tissue = attrs_pd.set_index("SAMPID")["SMTSD"]
sample_to_donor  = attrs_pd.set_index("SAMPID")["SUBJID"]
print("Tissue map loaded")

# ── 2. log2FC — gene_expression_matrix is small, pandas fine ──
gem = pd.read_parquet("gs://gene_datasets/gene_expression_matrix.parquet")
gem = gem.set_index('gene_id').drop(columns=['gene_name', 'gene_biotype'])
control_cols   = ['Control_1(HSR6)', 'Control_2(HSR6)']
control_mean   = gem[control_cols].mean(axis=1).clip(lower=1e-9)
treat_rep_cols = [c for c in gem.columns if c not in control_cols]
log2fc_raw = (
    gem[treat_rep_cols].div(control_mean, axis=0).clip(lower=1e-9).apply(np.log2)
)
log2fc_raw.columns = log2fc_raw.columns.str.split('_').str[0]
log2fc_by_treatment = (
    log2fc_raw.T.groupby(level=0).mean().T
    .replace([np.inf, -np.inf], np.nan).fillna(0)
)
pca_summary       = pd.read_csv('gs://gene_datasets/pca_summary.csv', index_col=0)
pareto_treatments = pca_summary[pca_summary['pareto_optimal'] == True].index.tolist()
log2fc_by_treatment = log2fc_by_treatment[
    [t for t in log2fc_by_treatment.columns if t in pareto_treatments]
]
print(f"log2FC shape: {log2fc_by_treatment.shape}")

# ── 3. Load GTEx in Spark — lazy, no memory hit at this line ─
gtex_spark  = spark.read.parquet("gs://gene_datasets/GTEx_tissue_expression.parquet")
sample_cols = [c for c in gtex_spark.columns if c.startswith("GTEX")]
print("read parquet")

# ── 4. Shared genes — reads only the Name column (fast) ──────
gtex_genes = (
    gtex_spark
    .select(F.regexp_replace("Name", r"\.\d+$", "").alias("gene_id"))
    .toPandas()["gene_id"]
)
shared_genes_set = set(log2fc_by_treatment.index) & set(gtex_genes)
log2fc_filtered  = log2fc_by_treatment.loc[log2fc_by_treatment.index.isin(shared_genes_set)]
print(f"Shared genes: {len(shared_genes_set)}")

# ── 5. Broadcast lookup dicts to Spark workers ───────────────
shared_genes_bc = spark.sparkContext.broadcast(shared_genes_set)
s2donor_bc      = spark.sparkContext.broadcast(sample_to_donor.to_dict())
s2tgroup_bc     = spark.sparkContext.broadcast(
    {s: t.split(" - ")[0].strip() for s, t in sample_to_tissue.items()}
)

# ── 6. Melt GTEx wide -> long inside Spark (mapInPandas) ─────
# Spark partitions by ROWS (genes).  Each pandas chunk is a subset of
# genes x all sample columns — processed and discarded immediately.
# The full 11 GB matrix is never in driver memory.
melt_schema = StructType([
    StructField("gene_id",      StringType(), False),
    StructField("donor",        StringType(), False),
    StructField("tissue_group", StringType(), False),
    StructField("expression",   FloatType(),  True),
])

def melt_partition(iterator):
    shared   = shared_genes_bc.value
    s2donor  = s2donor_bc.value
    s2tgroup = s2tgroup_bc.value
    for pdf in iterator:
        pdf["gene_id"] = pdf["Name"].str.replace(r"\.\d+$", "", regex=True)
        pdf = pdf[pdf["gene_id"].isin(shared)]
        if pdf.empty:
            continue
        sc = [c for c in pdf.columns if c.startswith("GTEX") and c in s2donor]
        if not sc:
            continue
        melted = pdf[["gene_id"] + sc].melt(
            id_vars="gene_id", value_vars=sc,
            var_name="sample", value_name="expression"
        )
        melted["expression"]   = melted["expression"].astype("float32")
        melted["donor"]        = melted["sample"].map(s2donor)
        melted["tissue_group"] = melted["sample"].map(s2tgroup)
        melted = melted.dropna(subset=["donor", "tissue_group"])
        yield melted[["gene_id", "donor", "tissue_group", "expression"]]

gtex_long = gtex_spark.mapInPandas(melt_partition, schema=melt_schema).cache()

# ── 7. Per-donor median expression per (gene, tissue_group) ──
donor_gene_tissue = (
    gtex_long
    .groupBy("gene_id", "donor", "tissue_group")
    .agg(F.percentile_approx("expression", 0.5).alias("median_expr"))
    .cache()
)

# ── 8. log2fc as a Spark DataFrame for the distributed dot product ─
log2fc_spark = spark.createDataFrame(
    log2fc_filtered.reset_index()
    .melt(id_vars="gene_id", var_name="treatment", value_name="log2fc")
    .astype({"log2fc": float})
)

# ── 9. Impact = sum_gene(log2fc x median_expr) per (donor, treatment, tissue) ─
impact_spark = (
    donor_gene_tissue
    .join(log2fc_spark, on="gene_id", how="inner")
    .groupBy("donor", "treatment", "tissue_group")
    .agg(F.sum(F.col("log2fc") * F.col("median_expr")).alias("impact_score"))
)

# ── 10. Collect — ~60K rows, tiny ────────────────────────────────
print("Computing impact scores in Spark (~2 min)...")
impact_long = impact_spark.toPandas()
print(f"Collected {len(impact_long)} rows")

# ── 11. Rebuild impact_by_donor (matches Cell C output format) ─
impact_by_donor = {}
for donor_id, grp in impact_long.groupby("donor"):
    mat = grp.pivot(index="treatment", columns="tissue_group", values="impact_score").fillna(0)
    mat.columns.name = None
    impact_by_donor[donor_id] = mat

print(f"Impact matrices computed for {len(impact_by_donor)} donors")
example = next(iter(impact_by_donor.values()))
print(f"Shape per donor (treatments x tissue groups): {example.shape}")

# ── 12. Stacked DataFrame (matches Cell D output format) ────────
stacked = (
    impact_long
    .pivot_table(index=["donor", "treatment"], columns="tissue_group",
                 values="impact_score", fill_value=0)
    .reset_index()
)
stacked.columns.name = None
tissue_cols = [c for c in stacked.columns if c not in ["donor", "treatment"]]
print(f"Stacked shape: {stacked.shape}")

# ── 13. Population-median GTEx (reuses cached donor_gene_tissue) ──
# Median-of-per-donor-medians avoids reprocessing the raw parquet.
print("Computing population-median GTEx matrix...")
gtex_pop_pd = (
    donor_gene_tissue
    .groupBy("gene_id", "tissue_group")
    .agg(F.percentile_approx("median_expr", 0.5).alias("median_expr"))
    .toPandas()
    .pivot(index="gene_id", columns="tissue_group", values="median_expr")
    .fillna(0)
)
gtex_pop_pd.columns.name = None
shared_agg = log2fc_filtered.index.intersection(gtex_pop_pd.index)
impact_agg = pd.DataFrame(
    log2fc_filtered.loc[shared_agg].values.T @ gtex_pop_pd.loc[shared_agg].values,
    index=log2fc_filtered.columns,
    columns=gtex_pop_pd.columns,
)
print(f"impact_agg shape: {impact_agg.shape}")

# ── 14. donor_groups (kept for compatibility with existing cells) ─
donor_groups = defaultdict(list)
for s in sample_cols:
    if s in sample_to_donor.index:
        donor_groups[sample_to_donor[s]].append(s)

print(f"\nDone. Skip to Cell E (ANOVA) for analysis.")

Tissue map loaded


/opt/conda/miniconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.8) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


log2FC shape: (78986, 4)


26/05/30 21:36:32 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/30 21:36:47 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/30 21:37:02 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/30 21:37:17 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/30 21:37:32 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/30 21:37:47 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registere

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from collections import defaultdict

# Step 1: Load tissue map
attrs_pd = pd.read_csv(
    "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
    sep="\t",
    usecols=["SAMPID", "SMTSD"]
)
attrs_pd["SUBJID"] = attrs_pd["SAMPID"].apply(
    lambda x: "-".join(x.split("-")[:2])
)
sample_to_tissue = attrs_pd.set_index("SAMPID")["SMTSD"]
sample_to_donor  = attrs_pd.set_index("SAMPID")["SUBJID"]
print("Tissue map loaded")

# Step 2: Read parquet directly with pandas — bypasses JVM entirely
print("Reading parquet... (may take 1-2 mins)")
df_pandas = pd.read_parquet(
    "gs://gene_datasets/GTEx_tissue_expression.parquet"
)
print(f"Loaded: {df_pandas.shape}")

In [ ]:
print("Should take ~30sec")
# Step 3: Clean index — Name is already the index
df_pandas.index = df_pandas.index.str.split(".").str[0]
df_pandas = df_pandas.drop(columns=["Description"])
print("Index cleaned")
print(df_pandas.shape)
print(df_pandas.index[:5].tolist())

# Step 4: Group samples by donor
sample_cols = [c for c in df_pandas.columns if c.startswith("GTEX")]
donor_groups = defaultdict(list)
for sample in sample_cols:
    if sample in sample_to_donor.index:
        donor_groups[sample_to_donor[sample]].append(sample)
print(f"Unique donors: {len(donor_groups)}")

# # Step 5: Build one dataframe per donor
# donor_dfs = {}
# for donor_id, samples in donor_groups.items():
#     donor_samples = [s for s in samples if s in df_pandas.columns]
#     donor_df = df_pandas[donor_samples].copy()
#     donor_df.columns = [sample_to_tissue[s] for s in donor_samples]
#     donor_df = donor_df.T.groupby(level=0).median().T
#     donor_dfs[donor_id] = donor_df

# print(f"\nTotal donor dataframes: {len(donor_dfs)}")
# print(f"Example donor shape: {donor_dfs[list(donor_dfs.keys())[0]].shape}")
# donor_dfs[list(donor_dfs.keys())[0]].head()

In [ ]:
import numpy as np

# Cell A — Load gene_expression_matrix.parquet (genes × treatment_replicates)
gem = pd.read_parquet("gs://gene_datasets/gene_expression_matrix.parquet")
gem = gem.set_index('gene_id').drop(columns=['gene_name', 'gene_biotype'])

# Control mean per gene — clip to avoid 0/0 = NaN for unexpressed genes
control_cols = ['Control_1(HSR6)', 'Control_2(HSR6)']
control_mean = gem[control_cols].mean(axis=1).clip(lower=1e-9)

# Log2FC for every non-control replicate
treat_rep_cols = [c for c in gem.columns if c not in control_cols]
log2fc_raw = (
    gem[treat_rep_cols]
    .div(control_mean, axis=0)
    .clip(lower=1e-9)
    .apply(np.log2)
)

# Group replicates by treatment type (prefix before first '_')
log2fc_raw.columns = log2fc_raw.columns.str.split('_').str[0]
log2fc_by_treatment = log2fc_raw.T.groupby(level=0).mean().T   # genes × treatments

# Replace any residual NaN/inf with 0
log2fc_by_treatment = (
    log2fc_by_treatment
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(f"log2FC matrix shape (genes × treatments): {log2fc_by_treatment.shape}")
print(f"Treatments: {log2fc_by_treatment.columns.tolist()}")
print(f"NaN count: {log2fc_by_treatment.isna().sum().sum()}")


In [ ]:
# Cell A2 — Filter to Pareto-optimal treatments only
pca_summary = pd.read_csv(
    'gs://gene_datasets/pca_summary.csv',
    index_col=0,
)

pareto_treatments = pca_summary[pca_summary['pareto_optimal'] == True].index.tolist()
print(f"All treatments: {log2fc_by_treatment.columns.tolist()}")
print(f"Pareto-optimal treatments ({len(pareto_treatments)}): {pareto_treatments}")

# Filter log2fc matrix to pareto-optimal treatments only
log2fc_by_treatment = log2fc_by_treatment[
    [t for t in log2fc_by_treatment.columns if t in pareto_treatments]
]
print(f"log2FC matrix after filter (genes × pareto treatments): {log2fc_by_treatment.shape}")

In [ ]:
print("Takes ~10sec")
# Cell B — Join CPM genes with GTEx ONCE in pandas before per-donor split
# Both now have Ensembl gene IDs as the index — direct intersection, no reshape needed
shared_genes = df_pandas.index.intersection(log2fc_by_treatment.index)
print(f"CPM genes:    {len(log2fc_by_treatment.index)}")
print(f"GTEx genes:   {len(df_pandas.index)}")
print(f"Shared genes: {len(shared_genes)}")

gtex_filtered   = df_pandas.loc[shared_genes]                # shared_genes × 19K samples
log2fc_filtered = log2fc_by_treatment.loc[shared_genes]      # shared_genes × treatments


In [ ]:
print("Takes about 1 min")
# Cell C — Per-donor impact matrix: (treatments × genes) @ (genes × tissues)
from joblib import Parallel, delayed

log2fc_vals = log2fc_filtered.values          # shape: (n_genes, n_treatments)
treatment_names = log2fc_filtered.columns.tolist()

def tissue_group(name):
    """'Brain - Cerebellar Hemisphere' → 'Brain', 'Thyroid' → 'Thyroid'"""
    return name.split(' - ')[0].strip()

def compute_impact(donor_id, samples):
    donor_samples = [s for s in samples if s in gtex_filtered.columns]
    if not donor_samples:
        return donor_id, None
    donor_gtex = gtex_filtered[donor_samples].copy()
    # Map samples → tissue group name (collapses sub-regions like "Brain - X" → "Brain")
    donor_gtex.columns = [tissue_group(sample_to_tissue[s]) for s in donor_samples]
    # Median across all samples in the same group (multiple brain regions, multiple samples)
    donor_gtex = donor_gtex.T.groupby(level=0).median().T
    donor_gtex = donor_gtex.fillna(0)
    impact = pd.DataFrame(
        log2fc_vals.T @ donor_gtex.values,
        index=treatment_names,
        columns=donor_gtex.columns,
    )
    return donor_id, impact

results = Parallel(n_jobs=-1, prefer='threads')(
    delayed(compute_impact)(did, samps)
    for did, samps in donor_groups.items()
)
impact_by_donor = {did: imp for did, imp in results if imp is not None}

print(f"Impact matrices computed for {len(impact_by_donor)} donors")
example = next(iter(impact_by_donor.values()))
print(f"Shape per donor (treatments × tissue groups): {example.shape}")
print(f"Tissue groups: {example.columns.tolist()}")
print(f"Sample values:\n{example.head(3)}")


In [ ]:
# Cell D — Stack all donors into one DataFrame
# Different donors have different tissues sampled — fill missing with 0
# (no sample for a tissue means no measured impact, not NaN)
import plotly.express as px

stacked = pd.concat(
    impact_by_donor,
    names=['donor', 'treatment'],
).fillna(0).reset_index()

tissue_cols = [c for c in stacked.columns if c not in ['donor', 'treatment']]
print(f"Stacked shape: {stacked.shape}")
print(f"Tissues ({len(tissue_cols)}): {tissue_cols}")
print(f"NaN count: {stacked.isna().sum().sum()}")
stacked.head()

In [ ]:
# Cell E — ANOVA per tissue: which tissues differ most across treatment types?
from scipy.stats import f_oneway

anova_results = {}
for tissue in tissue_cols:
    groups = [g[tissue].values for _, g in stacked.groupby('treatment')]
    stat, p = f_oneway(*groups)
    anova_results[tissue] = {'F_stat': stat, 'p_value': p}

anova_df = (
    pd.DataFrame(anova_results).T
    .sort_values('F_stat', ascending=False)
    .astype(float)
)

print("Top tissues by F-statistic (most explained by treatment):")
print(anova_df.head(30).to_string())

px.bar(
    anova_df.reset_index(),
    x='index', y='F_stat',
    labels={'index': 'Tissue', 'F_stat': 'F-Statistic (ANOVA)'},
    title='ANOVA: Which Tissues Are Most Explained by Treatment Type?'
          '<br><sup>Higher F-stat = treatment type explains more variance in tissue impact score</sup>',
).update_layout(xaxis_tickangle=-45).show()


In [ ]:
# Cell F — Random Forest feature importance: nonlinear cross-check of ANOVA ranking
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

X = stacked[tissue_cols].fillna(0).values
y = LabelEncoder().fit_transform(stacked['treatment'])

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = (
    pd.DataFrame({'tissue': tissue_cols, 'importance': rf.feature_importances_})
    .sort_values('importance', ascending=False)
)

px.bar(
    importance_df,
    x='tissue', y='importance',
    labels={'tissue': 'Tissue', 'importance': 'Feature Importance'},
    title='RF Feature Importance: Which Tissues Best Classify Treatment Type?'
          '<br><sup>Higher = tissue impact score best discriminates between treatments</sup>',
).update_layout(xaxis_tickangle=-45).show()


In [ ]:
# Cell G — Compare ANOVA vs RF rankings side-by-side
# Rank each method (1 = most important tissue)
anova_rank = anova_df['F_stat'].rank(ascending=False).rename('ANOVA rank')
rf_rank = importance_df.set_index('tissue')['importance'].rank(ascending=False).rename('RF rank')

rank_df = pd.concat([anova_rank, rf_rank], axis=1).sort_values('ANOVA rank')

fig = px.scatter(
    rank_df.reset_index(),
    x='ANOVA rank', y='RF rank',
    text='index',
    title='ANOVA vs RF Tissue Rankings'
          '<br><sup>Points near the diagonal agree between methods; outliers suggest nonlinear interactions</sup>',
    labels={'index': 'Tissue'},
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_shape(type='line', x0=1, y0=1, x1=len(tissue_cols), y1=len(tissue_cols),
              line=dict(dash='dash', color='gray'))
fig.show()

print("\nFull ranking comparison:")
print(rank_df.to_string())


In [ ]:
# Mean tissue impact per treatment (averaged across all 946 donors)
treatment_tissue_mean = stacked.groupby('treatment')[tissue_cols].mean()

fig_heat = px.imshow(
    treatment_tissue_mean,
    labels=dict(x='Tissue Group', y='Treatment', color='Mean Impact Score'),
    title='Mean Tissue Group Impact by Treatment'
          '<br><sup>Impact = Σ(log2FC × GTEx expression) across shared genes. '
          'Positive = treatment upregulates genes active in this tissue; '
          'Negative = downregulates.</sup>',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    aspect='auto',
)
fig_heat.update_layout(
    xaxis_tickangle=-45,
    coloraxis_colorbar=dict(title='Impact Score'),
)
fig_heat.show()

# Ranked tissue per treatment — most impacted tissue for each treatment
print("Most impacted tissue per treatment (by absolute mean impact):")
print(
    treatment_tissue_mean.abs()
    .idxmax(axis=1)
    .rename('top_tissue')
    .to_frame()
    .join(
        treatment_tissue_mean.abs().max(axis=1).rename('abs_impact_score')
    )
    .sort_values('abs_impact_score', ascending=False)
    .to_string()
)


---
## Aggregated Analysis — Single Population-Median GTEx Matrix

Instead of computing one impact matrix per donor (946 iterations), this collapses all GTEx samples to a single **population-median** expression value per gene per tissue group first, then computes the impact in one matrix multiply. The result is one row per treatment — cleaner, faster, and removes donor noise from the signal entirely.

In [ ]:
# Aggregate all GTEx samples to one population-median per tissue group
# Avoids copying gtex_filtered to reduce memory pressure

valid_samples = [s for s in gtex_filtered.columns if s in sample_to_tissue.index]

# Rename columns in-place on a view, then groupby — no full copy of the 11GB matrix
gtex_agg = gtex_filtered[valid_samples]
gtex_agg = gtex_agg.rename(
    columns={s: tissue_group(sample_to_tissue[s]) for s in valid_samples}
)

# Median across ALL samples in the same tissue group
gtex_median = gtex_agg.T.groupby(level=0).median().T.fillna(0)  # genes × tissue_groups
del gtex_agg  # free memory immediately

print(f"Population-median GTEx shape (genes × tissue groups): {gtex_median.shape}")
print(f"Tissue groups: {gtex_median.columns.tolist()}")

# Single matrix multiply: one impact score per treatment per tissue
impact_agg = pd.DataFrame(
    log2fc_filtered.values.T @ gtex_median.values,
    index=log2fc_filtered.columns,
    columns=gtex_median.columns,
)
del gtex_median  # free memory immediately

print(f"\nAggregated impact shape (treatments × tissue groups): {impact_agg.shape}")
print(impact_agg.to_string())


In [ ]:
# Heatmap and ranked table for aggregated analysis
fig_agg = px.imshow(
    impact_agg,
    labels=dict(x='Tissue Group', y='Treatment', color='Impact Score'),
    title='Aggregated Impact: Treatment × Tissue (Population-Median GTEx)'
          '<br><sup>One score per cell — donor noise removed. '
          'Negative = treatment downregulates genes highly expressed in this tissue.</sup>',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    aspect='auto',
)
fig_agg.update_layout(xaxis_tickangle=-45)
fig_agg.show()

print("Most impacted tissue per treatment (aggregated):")
print(
    impact_agg.abs()
    .idxmax(axis=1)
    .rename('top_tissue')
    .to_frame()
    .join(impact_agg.abs().max(axis=1).rename('abs_impact_score'))
    .sort_values('abs_impact_score', ascending=False)
    .to_string()
)


---
## Plan: Does Treatment Tissue Impact Vary Significantly Across Donors?

The `stacked` DataFrame already contains one impact score per (donor, treatment, tissue), so the data needed for this question is already computed. Here is a step-by-step plan:

**Step 1 — Coefficient of Variation (CV) heatmap**
For each (treatment, tissue) pair, compute `std / |mean|` across donors. A high CV means the tissue impact is unreliable — it depends heavily on who the patient is. Visualize as a heatmap alongside the mean impact heatmap. Tissues where CV is high relative to the mean signal are the ones where donor biology dominates. Important: exclude donors where a tissue wasn't sampled (the zeros from `fillna(0)`) — otherwise CV is artificially inflated. This requires tracking which entries were originally NaN before filling.

**Step 2 — Link GTEx donor metadata**
GTEx provides subject-level phenotypes in `GTEx_Analysis_v11_Annotations_SubjectPhenotypesDS.txt` (on GCS). Columns include `SUBJID`, `SEX` (1=male, 2=female), `AGE` (decade bins: 20-29, 30-39, ..., 70-79), and `DTHHRDY` (death circumstances). Merge this with `stacked` on the `donor` column to add these covariates.

**Step 3 — Linear regression per (treatment, tissue)**
For each (treatment, tissue) pair, regress impact score ~ sex + age using `scipy.stats.linregress` or `statsmodels.OLS`. The coefficient on sex tells you: "for this treatment in this tissue, do males and females show different impact?" Report as a heatmap of regression coefficients and p-values. Apply Bonferroni correction for the number of tests (n_treatments × n_tissues).

**Step 4 — Variance decomposition**
Use a mixed-effects model (`statsmodels.MixedLM`) with the formula `impact ~ treatment + (1|donor)`. The intraclass correlation coefficient (ICC) tells you what fraction of total variance is explained by donor identity vs. treatment. If ICC is high (e.g., >0.3), donor biology is a major confounder and the per-donor analysis is warranted. If ICC is low, the aggregated median result is sufficient.

**Step 5 — Interpret**
Cross-reference the CV heatmap with the mean impact heatmap:
- High mean + low CV → reliable signal; this tissue is consistently impacted across donors
- High mean + high CV → donor-dependent; worth stratifying by sex/age
- Low mean + any CV → tissue is not meaningfully impacted by this treatment


In [ ]:
# # Cell 1 — kill the existing session
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
# spark.stop()
# print("Stopped")

In [ ]:
# # load packages
# from pyspark.sql import SparkSession
# import pandas as pd
# from pyspark.sql import functions as F

# # Cell 2 — start fresh in local mode
# spark = SparkSession.builder \
#     .appName("GTEx Full Load") \
#     .master("local[4]") \
#     .config("spark.driver.memory", "24g") \
#     .config("spark.sql.parquet.mergeSchema", "false") \
#     .config("spark.sql.parquet.filterPushdown", "true") \
#     .config("spark.sql.shuffle.partitions", "4") \
#     .config("spark.driver.maxResultSize", "8g") \
#     .getOrCreate()

# print(spark.sparkContext.master)  # should print "local[4]"

In [ ]:
# # load data into memory (4GB takes ~30 sec)
# df = spark.read.parquet("gs://gene_datasets/GTEx_tissue_expression.parquet")
# print((df.count(), len(df.columns)))

In [ ]:
# # Read sample attributes to link codes to tissues
# # Step 2: Load tissue attributes
# attrs = spark.createDataFrame(
#     pd.read_csv(
#         "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
#         sep="\t",
#         usecols=["SAMPID", "SMTSD"]
#     )
# )

In [ ]:
# import pandas as pd
# from collections import defaultdict

# # Step 1: Load tissue map and extract donor ID
# attrs_pd = pd.read_csv(
#     "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
#     sep="\t",
#     usecols=["SAMPID", "SMTSD"]
# )
# attrs_pd["SUBJID"] = attrs_pd["SAMPID"].apply(
#     lambda x: "-".join(x.split("-")[:2])
# )
# sample_to_tissue = attrs_pd.set_index("SAMPID")["SMTSD"]
# sample_to_donor  = attrs_pd.set_index("SAMPID")["SUBJID"]

# # Step 2: Get sample columns and group by donor
# sample_cols = [c for c in df.columns if c.startswith("GTEX")]
# donor_groups = defaultdict(list)
# for sample in sample_cols:
#     if sample in sample_to_donor.index:
#         donor_id = sample_to_donor[sample]
#         donor_groups[donor_id].append(sample)

# print(f"Unique donors: {len(donor_groups)}")

# # Step 3: Convert Spark df to pandas and clean index
# df_pandas = df.toPandas().set_index("Name")
# df_pandas.index = df_pandas.index.str.split(".").str[0]
# df_pandas = df_pandas.drop(columns=["Description"])

# # Step 4: Build one dataframe per donor
# donor_dfs = {}
# for donor_id, samples in donor_groups.items():
#     donor_samples = [s for s in samples if s in df_pandas.columns]
#     donor_df = df_pandas[donor_samples].copy()
#     donor_df.columns = [sample_to_tissue[s] for s in donor_samples]
#     # If donor has multiple samples per tissue, take median
#     donor_df = donor_df.T.groupby(level=0).median().T
#     donor_dfs[donor_id] = donor_df

# print(f"Total donor dataframes: {len(donor_dfs)}")
# print(f"\nExample donor shape: {donor_dfs[list(donor_dfs.keys())[0]].shape}")
# donor_dfs[list(donor_dfs.keys())[0]].head()